# 09 — Joint text/image indexing and retrieval

**Milestone 2b.** This notebook demonstrates how normalized local assets become image vectors, how the vectors are associated with catalog items, and how queries are scored and evaluated.

The **offline cells use a deliberately simple test double**, not a trained model. It uses image color statistics and two text keywords only to exercise the pipeline. The optional final section runs the actual CLIP adapter with locally cached model weights. No model downloads occur when this notebook is run in CI.

The two example illustrations and relevance labels are synthetic. The two image-to-image cases are self-matches; their scores are pipeline checks, **not retrieval generalization results**.

## 1. Prepare a local image catalog

`media ingest` normalizes the source illustrations and writes a self-contained catalog. All work takes place in a temporary directory; the repository is not modified.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

from savoroute.data.catalog import load_catalog
from savoroute.media.ingestion import prepare_image_batch, write_image_batch

root = Path.cwd()
assert (root / "data/sample/image_manifest.json").is_file(), "Launch from the repository root"
samples = root / "data/sample"
workspace = TemporaryDirectory(prefix="savoroute-vectors-")
media_dir = Path(workspace.name) / "media"
batch = prepare_image_batch(samples / "image_manifest.json", load_catalog(samples / "catalog.json"))
write_image_batch(batch, media_dir)
catalog = load_catalog(media_dir / "catalog.json")
print(f"Items: {len(catalog.items)}, images: {len(catalog.media)}")


## 2. Exercise the index with a deterministic test double

The test double maps the two image illustrations to orthogonal vectors using a pixel statistic. It recognizes keywords in text queries. It demonstrates **vector-index behavior only**; it cannot test learned image/text alignment.

In [ ]:
from PIL import Image, ImageStat

class DemoEncoder:
    model_id = "offline-demonstration"
    embedding_space_id = "offline-demonstration@1"

    def embed_texts(self, texts):
        return [[0.0, 1.0] if "mushroom" in text.casefold() else [1.0, 0.0]
                for text in texts]

    def embed_images(self, paths):
        output = []
        for path in paths:
            with Image.open(path) as image:
                mean_green = ImageStat.Stat(image.convert("RGB")).mean[1]
            output.append([0.0, 1.0] if mean_green > 190 else [1.0, 0.0])
        return output

encoder = DemoEncoder()


In [ ]:
from savoroute.retrieval.vector import (
    build_vector_index, index_to_dict, load_vector_index,
    search_with_encoder, validate_catalog, write_vector_index,
)

index = build_vector_index(catalog, media_dir, encoder)
index_path = Path(workspace.name) / "vectors.json"
write_vector_index(index, index_path)
loaded = load_vector_index(index_path)
validate_catalog(loaded, catalog)
print("Embedding space:", loaded.embedding_space_id)
print("Dimensions:", loaded.dimension)
print("Indexed records:", len(loaded.records))
print("Index keys:", list(index_to_dict(loaded)))


## 3. Text-to-image and image-to-image queries

Similarity is a dot product of unit-normalized vectors (cosine similarity). The index preserves modality and item IDs. Image records are evaluated as separate candidates, not implicitly merged with text records.

In [ ]:
for query in ("orange noodles", "mushroom pasta"):
    hits = search_with_encoder(loaded, encoder, text=query, target="image", k=2,
                               catalog=catalog)
    print(query, [(hit.record_id, round(hit.score, 3)) for hit in hits])

image_query = media_dir / "assets/image_p002.png"
image_hits = search_with_encoder(loaded, encoder, image=image_query, target="image", k=2)
print("Image query:", [(hit.record_id, round(hit.score, 3)) for hit in image_hits])


## 4. Labeled evaluation

Benchmark judgments are explicit. Mean Recall@k, reciprocal rank, and nDCG are computed over the four fictional cases. Image self-matches are counted separately and are not evidence of generalizable image retrieval.

In [ ]:
from savoroute.evaluation.multimodal import (
    evaluate_multimodal, load_multimodal_benchmark,
)

benchmark = load_multimodal_benchmark(samples / "multimodal_queries_fixture.json")
report = evaluate_multimodal(loaded, encoder, benchmark, k=1)
print("Queries:", len(report.cases), "Self-match cases:", report.self_match_cases)
print("Recall@1:", report.mean_recall_at_k, "MRR@1:", report.mrr_at_k)
assert report.mean_recall_at_k == 1.0  # only validates the synthetic test double


## 5. Optional: run the actual CLIP encoder

Set `RUN_CLIP = True` after installing `savoroute[multimodal]` and caching the model weights, or pass `local_files_only=False` explicitly if you intend to download them. For reproducible measurements, pin the **model revision to a full commit hash** and record the dependency environment. The production index uses the exact same build, search, persistence, and evaluation functions shown above.

In [ ]:
RUN_CLIP = False
if RUN_CLIP:
    from savoroute.providers.clip import ClipTextImageEmbedder

    clip = ClipTextImageEmbedder(
        model_id="sentence-transformers/clip-ViT-B-32",
        revision=None,  # set to a specific model commit for reproducible experiments
        local_files_only=True,
    )
    actual_index = build_vector_index(catalog, media_dir, clip)
    actual_hits = search_with_encoder(actual_index, clip, text="mushroom pasta",
                                      target="image", k=2, catalog=catalog)
    print("CLIP retrieval:", [(h.record_id, round(h.score, 3)) for h in actual_hits])
    actual_score = evaluate_multimodal(actual_index, clip, benchmark, k=2)
    print(actual_score.as_dict())


The next milestone will introduce larger-scale vector storage and item-level multimodal fusion. This implementation intentionally uses a small exact-search JSON index, so results can be inspected and independently reproduced.

In [ ]:
workspace.cleanup()